# NorthStar Urban Mobility - MongoDB Development
## Part 4: NoSQL Document Modelling, CRUD Operations & Aggregation Pipelines

This notebook demonstrates how NorthStar's semi-structured and event-driven data (app events, complaints, incidents) can be reshaped from flat relational CSVs into rich, nested MongoDB documents. It uses PyMongo to connect to MongoDB Atlas and performs CRUD operations and aggregation pipelines.

### Why MongoDB for NorthStar?
The case study identifies that NorthStar's relational schema cannot efficiently handle:
- **Semi-structured app event streams** with variable fields per event type
- **Nested customer journeys** combining orders, complaints, and app interactions
- **Flexible incident documents** with varying attributes per incident type

MongoDB's document model allows embedding related data together, reducing expensive JOINs and providing a natural structure for these use cases.

## Setup
Run this cell first to clone the data repository and install PyMongo.

**Update the `MONGO_URI` in the next code cell with your Atlas connection string.**

In [ ]:
import os
if not os.path.exists('northstar-coursework'):
    !git clone https://github.com/Erucard/northstar-coursework.git
!pip install -q pymongo[srv] dnspython
os.chdir('/content/northstar-coursework/data/raw')
print('Ready:', sorted([f for f in os.listdir('.') if f.endswith('.csv')]))
print('JSON:', sorted(os.listdir('../json/')))

In [ ]:
import pymongo
from pymongo import MongoClient, ASCENDING, DESCENDING
import pandas as pd
import json
from datetime import datetime
from pprint import pprint

# ============================================================
# MONGODB ATLAS CONNECTION
# ============================================================
# Replace with your MongoDB Atlas connection string
MONGO_URI = 'mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority'

client = MongoClient(MONGO_URI)
db = client['northstar_mobility']

# Verify connection
print('Connected to MongoDB Atlas')
print('Database:', db.name)
print('Server info:', client.server_info()['version'])

In [ ]:
# ============================================================
# DATA LOADING AND ZONE STANDARDISATION
# ============================================================

data_path = '.'

customers = pd.read_csv(f'{data_path}/customers.csv')
orders = pd.read_csv(f'{data_path}/orders.csv')
deliveries = pd.read_csv(f'{data_path}/deliveries.csv')
complaints = pd.read_csv(f'{data_path}/complaints.csv')
incidents = pd.read_csv(f'{data_path}/incidents.csv')
app_events = pd.read_csv(f'{data_path}/app_events.csv')
drivers = pd.read_csv(f'{data_path}/drivers.csv')
vehicles = pd.read_csv(f'{data_path}/vehicles.csv')
hubs = pd.read_csv(f'{data_path}/hubs.csv')

zone_map = {
    'north': 'North', 'NORTH': 'North',
    'south': 'South', 'SOUTH': 'South',
    'east': 'East', 'EAST': 'East',
    'west': 'West', 'WEST': 'West',
    'central': 'Central', 'CENTRAL': 'Central', 'Ctr': 'Central',
    'airport': 'Airport', 'AIRPORT': 'Airport',
    'riverside': 'Riverside', 'RiverSide': 'Riverside'
}

customers['home_zone'] = customers['home_zone'].replace(zone_map)
orders['pickup_zone'] = orders['pickup_zone'].replace(zone_map)
orders['dropoff_zone'] = orders['dropoff_zone'].replace(zone_map)
app_events['zone_context'] = app_events['zone_context'].replace(zone_map)
drivers['base_zone'] = drivers['base_zone'].replace(zone_map)
vehicles['assigned_zone'] = vehicles['assigned_zone'].replace(zone_map)

print('Data loaded and zones standardised.')

## Document Schema Design

### Design Rationale

We create **three collections** with purposeful document structures:

1. **`customer_journeys`** — Customer-centric documents with embedded orders, complaints, and app events. This eliminates the need for multi-table JOINs when analysing a customer's full interaction history.

2. **`delivery_operations`** — Delivery-centric documents embedding driver info, vehicle info, hub info, and linked incidents. This supports the operations team's need to see the full context of each delivery in one query.

3. **`app_event_stream`** — Flattened event documents optimised for time-series analysis and aggregation pipelines. Each event is a standalone document with denormalised customer and zone data.

### Embedding vs Referencing Decision
- **Embedded**: Complaints within customer journeys (1:few relationship, always queried together)
- **Embedded**: Incidents within delivery operations (1:few, context-critical)
- **Referenced**: Orders are embedded as summaries, with order_id for cross-referencing
- **Standalone**: App events are kept as individual documents for time-series aggregation efficiency

In [ ]:
# ============================================================
# COLLECTION 1: CUSTOMER JOURNEYS
# Document structure: one document per customer with embedded
# orders, complaints, and app event summaries.
# ============================================================

# Drop existing collection for clean run
db.customer_journeys.drop()

def build_customer_document(cust_row):
    """Build a customer-centric document with embedded sub-documents."""
    cid = cust_row['customer_id']
    
    # Get this customer's orders
    cust_orders = orders[orders['customer_id'] == cid]
    order_docs = []
    for _, o in cust_orders.iterrows():
        order_doc = {
            'order_id': o['order_id'],
            'service_type': o['service_type'],
            'order_created_at': o['order_created_at'],
            'pickup_zone': o['pickup_zone'],
            'dropoff_zone': o['dropoff_zone'],
            'priority_level': o['priority_level'],
            'order_value': float(o['order_value']),
            'booking_channel': o['booking_channel'] if pd.notna(o['booking_channel']) else None,
            'special_handling': bool(o['special_handling_flag'])
        }
        order_docs.append(order_doc)
    
    # Get this customer's complaints
    cust_complaints = complaints[complaints['customer_id'] == cid]
    complaint_docs = []
    for _, cp in cust_complaints.iterrows():
        complaint_doc = {
            'complaint_id': cp['complaint_id'],
            'order_id': cp['order_id'],
            'complaint_type': cp['complaint_type'],
            'channel': cp['channel'],
            'severity': cp['severity'],
            'created_at': cp['created_at'],
            'status': cp['status'],
            'resolution_days': int(cp['resolution_days']),
            'compensation_amount': float(cp['compensation_amount']) if pd.notna(cp['compensation_amount']) else 0.0
        }
        complaint_docs.append(complaint_doc)
    
    # Get this customer's app events
    cust_events = app_events[app_events['customer_id'] == cid]
    event_summary = {
        'total_events': len(cust_events),
        'event_types': cust_events['event_type'].value_counts().to_dict() if len(cust_events) > 0 else {},
        'avg_latency_ms': float(cust_events['api_latency_ms'].mean()) if len(cust_events) > 0 else None,
        'success_rate': float(cust_events['success_flag'].mean()) if len(cust_events) > 0 else None,
        'devices_used': list(cust_events['device_type'].unique()) if len(cust_events) > 0 else []
    }
    
    # Build the full customer document
    doc = {
        'customer_id': cid,
        'profile': {
            'age': int(cust_row['age']),
            'home_zone': cust_row['home_zone'],
            'customer_type': cust_row['customer_type'],
            'signup_date': cust_row['signup_date'],
            'account_status': cust_row['account_status']
        },
        'engagement': {
            'loyalty_score': float(cust_row['loyalty_score']) if pd.notna(cust_row['loyalty_score']) else None,
            'app_engagement_score': float(cust_row['app_engagement_score']),
            'preferred_channel': cust_row['preferred_channel'] if pd.notna(cust_row['preferred_channel']) else None
        },
        'orders': order_docs,
        'complaints': complaint_docs,
        'app_activity': event_summary,
        'metrics': {
            'total_orders': len(order_docs),
            'total_complaints': len(complaint_docs),
            'total_order_value': sum(o['order_value'] for o in order_docs),
            'total_compensation': sum(c['compensation_amount'] for c in complaint_docs),
            'is_repeat_complainer': len(complaint_docs) >= 2
        }
    }
    return doc

# Build and insert all customer documents
customer_docs = [build_customer_document(row) for _, row in customers.iterrows()]
result = db.customer_journeys.insert_many(customer_docs)

print(f'Inserted {len(result.inserted_ids)} customer journey documents')

# Show example document structure
print('\n=== Example Document Structure ===')
example = db.customer_journeys.find_one({'metrics.total_complaints': {'$gte': 2}})
pprint(example, width=100)

In [ ]:
# ============================================================
# COLLECTION 2: DELIVERY OPERATIONS
# Document structure: one document per delivery with embedded
# driver, vehicle, hub context, and incidents.
# ============================================================

db.delivery_operations.drop()

def build_delivery_document(del_row):
    """Build a delivery-centric document with embedded context."""
    did = del_row['delivery_id']
    
    # Get order info
    order_match = orders[orders['order_id'] == del_row['order_id']]
    order_info = {}
    if len(order_match) > 0:
        o = order_match.iloc[0]
        order_info = {
            'order_id': o['order_id'],
            'service_type': o['service_type'],
            'pickup_zone': o['pickup_zone'],
            'dropoff_zone': o['dropoff_zone'],
            'priority_level': o['priority_level'],
            'order_value': float(o['order_value'])
        }
    
    # Get driver info
    driver_match = drivers[drivers['driver_id'] == del_row['driver_id']]
    driver_info = {}
    if len(driver_match) > 0:
        dr = driver_match.iloc[0]
        driver_info = {
            'driver_id': dr['driver_id'],
            'base_zone': dr['base_zone'],
            'employment_type': dr['employment_type'],
            'rating': float(dr['driver_rating']),
            'training_score': float(dr['training_score']) if pd.notna(dr['training_score']) else None
        }
    
    # Get vehicle info
    veh_match = vehicles[vehicles['vehicle_id'] == del_row['vehicle_id']]
    vehicle_info = {}
    if len(veh_match) > 0:
        v = veh_match.iloc[0]
        vehicle_info = {
            'vehicle_id': v['vehicle_id'],
            'vehicle_type': v['vehicle_type'],
            'battery_health_pct': float(v['battery_health_pct']) if pd.notna(v['battery_health_pct']) else None,
            'maintenance_status': v['maintenance_status']
        }
    
    # Get hub info
    hub_match = hubs[hubs['hub_id'] == del_row['hub_id']]
    hub_info = {}
    if len(hub_match) > 0:
        h = hub_match.iloc[0]
        hub_info = {
            'hub_id': h['hub_id'],
            'hub_name': h['hub_name'],
            'zone': h['zone'],
            'hub_type': h['hub_type'],
            'capacity_score': int(h['capacity_score'])
        }
    
    # Get incidents
    del_incidents = incidents[incidents['delivery_id'] == did]
    incident_docs = []
    for _, inc in del_incidents.iterrows():
        incident_docs.append({
            'incident_id': inc['incident_id'],
            'incident_type': inc['incident_type'],
            'severity': inc['severity'],
            'resolution_status': inc['resolution_status'],
            'resolved_hours': float(inc['resolved_hours']) if pd.notna(inc['resolved_hours']) else None
        })
    
    # Compute delivery hours
    delivery_hours = None
    if pd.notna(del_row['delivery_completed_at']):
        try:
            dt_dispatch = pd.to_datetime(del_row['dispatch_time'])
            dt_complete = pd.to_datetime(del_row['delivery_completed_at'])
            delivery_hours = (dt_complete - dt_dispatch).total_seconds() / 3600
        except:
            pass
    
    doc = {
        'delivery_id': did,
        'delivery_status': del_row['delivery_status'],
        'dispatch_time': del_row['dispatch_time'],
        'delivery_completed_at': del_row['delivery_completed_at'] if pd.notna(del_row['delivery_completed_at']) else None,
        'delivery_hours': round(delivery_hours, 2) if delivery_hours else None,
        'route': {
            'distance_km': float(del_row['route_distance_km']),
            'manual_override_count': int(del_row['manual_route_override_count']),
            'fuel_or_charge_cost': float(del_row['fuel_or_charge_cost']),
            'cost_per_km': round(float(del_row['fuel_or_charge_cost']) / float(del_row['route_distance_km']), 3)
        },
        'quality': {
            'proof_of_completion_missing': bool(del_row['proof_of_completion_missing']),
            'customer_rating': float(del_row['customer_rating_post_delivery']) if pd.notna(del_row['customer_rating_post_delivery']) else None
        },
        'order': order_info,
        'driver': driver_info,
        'vehicle': vehicle_info,
        'hub': hub_info,
        'incidents': incident_docs,
        'has_incidents': len(incident_docs) > 0,
        'data_quality_flags': {
            'negative_delivery_time': (delivery_hours is not None and delivery_hours < 0),
            'missing_completion_time': pd.isna(del_row['delivery_completed_at'])
        }
    }
    return doc

delivery_docs = [build_delivery_document(row) for _, row in deliveries.iterrows()]
result = db.delivery_operations.insert_many(delivery_docs)

print(f'Inserted {len(result.inserted_ids)} delivery operation documents')

# Show example
print('\n=== Example Delivery Document (with incidents) ===')
example_del = db.delivery_operations.find_one({'has_incidents': True, 'delivery_status': 'Failed'})
pprint(example_del, width=100)

In [ ]:
# ============================================================
# COLLECTION 3: APP EVENT STREAM
# Document structure: one document per event, denormalised
# with customer and zone context for aggregation efficiency.
# ============================================================

db.app_event_stream.drop()

# Build a lookup for customer zone and type
cust_lookup = customers.set_index('customer_id')[['home_zone', 'customer_type']].to_dict('index')

event_docs = []
for _, ev in app_events.iterrows():
    cust_info = cust_lookup.get(ev['customer_id'], {})
    doc = {
        'event_id': ev['event_id'],
        'customer_id': ev['customer_id'],
        'customer_zone': cust_info.get('home_zone', 'Unknown'),
        'customer_type': cust_info.get('customer_type', 'Unknown'),
        'order_id': ev['order_id'] if pd.notna(ev['order_id']) else None,
        'event_timestamp': ev['event_timestamp'],
        'event_type': ev['event_type'],
        'session_id': ev['session_id'],
        'device_type': ev['device_type'],
        'zone_context': ev['zone_context'],
        'api_latency_ms': int(ev['api_latency_ms']),
        'success': bool(ev['success_flag']),
        'is_high_latency': ev['api_latency_ms'] > 500
    }
    event_docs.append(doc)

result = db.app_event_stream.insert_many(event_docs)
print(f'Inserted {len(result.inserted_ids)} app event documents')

In [ ]:
# ============================================================
# CREATE INDEXES
# ============================================================

# Customer journeys indexes
db.customer_journeys.create_index('customer_id', unique=True)
db.customer_journeys.create_index('profile.home_zone')
db.customer_journeys.create_index('metrics.total_complaints')
db.customer_journeys.create_index('engagement.loyalty_score')

# Delivery operations indexes
db.delivery_operations.create_index('delivery_id', unique=True)
db.delivery_operations.create_index('delivery_status')
db.delivery_operations.create_index('hub.hub_id')
db.delivery_operations.create_index('order.pickup_zone')
db.delivery_operations.create_index([('delivery_status', ASCENDING), ('hub.hub_id', ASCENDING)])

# App event stream indexes
db.app_event_stream.create_index('event_type')
db.app_event_stream.create_index('customer_id')
db.app_event_stream.create_index([('event_type', ASCENDING), ('success', ASCENDING)])

print('Indexes created successfully.')
print('\ncustomer_journeys indexes:', list(db.customer_journeys.index_information().keys()))
print('delivery_operations indexes:', list(db.delivery_operations.index_information().keys()))
print('app_event_stream indexes:', list(db.app_event_stream.index_information().keys()))

In [ ]:
# ============================================================
# CRUD OPERATIONS
# ============================================================

# --- READ: Find Operations ---

# R1: Find repeat complainers with high loyalty scores (paradox customers)
print('=== READ 1: High-Loyalty Repeat Complainers ===')
paradox_customers = db.customer_journeys.find(
    {
        'metrics.is_repeat_complainer': True,
        'engagement.loyalty_score': {'$gte': 50}
    },
    {
        'customer_id': 1,
        'engagement.loyalty_score': 1,
        'metrics.total_complaints': 1,
        'metrics.total_order_value': 1,
        '_id': 0
    }
).sort('metrics.total_complaints', DESCENDING).limit(10)

for doc in paradox_customers:
    pprint(doc)

print('\nThese are high-value customers at risk of churn despite loyalty.')

In [ ]:
# R2: Find failed deliveries with vehicle faults at Central zone hubs
print('=== READ 2: Failed Deliveries with Vehicle Faults (Central Zone) ===')
central_failures = db.delivery_operations.find(
    {
        'delivery_status': 'Failed',
        'hub.zone': 'Central',
        'incidents': {'$elemMatch': {'incident_type': {'$in': ['VehicleFault', 'BatteryAlert']}}}
    },
    {
        'delivery_id': 1,
        'hub.hub_name': 1,
        'vehicle.vehicle_id': 1,
        'vehicle.battery_health_pct': 1,
        'incidents.incident_type': 1,
        'incidents.severity': 1,
        '_id': 0
    }
).limit(5)

for doc in central_failures:
    pprint(doc)

print('\nShows how MongoDB can retrieve delivery + vehicle + incident context in one query.')

In [ ]:
# R3: Find app events with high latency grouped by failure
print('=== READ 3: Failed High-Latency App Events ===')
failed_events = db.app_event_stream.find(
    {
        'success': False,
        'is_high_latency': True
    }
).sort('api_latency_ms', DESCENDING).limit(5)

for doc in failed_events:
    pprint(doc)

In [ ]:
# --- UPDATE Operations ---

# U1: Flag high-risk customers (repeat complainers with low loyalty)
print('=== UPDATE 1: Flag High-Risk Customers ===')
update_result = db.customer_journeys.update_many(
    {
        'metrics.is_repeat_complainer': True,
        'engagement.loyalty_score': {'$lt': 40}
    },
    {
        '$set': {
            'risk_flag': 'HIGH_CHURN_RISK',
            'flagged_at': datetime.now().isoformat()
        }
    }
)
print(f'Matched: {update_result.matched_count}, Modified: {update_result.modified_count}')

# U2: Add data quality warning to deliveries with negative times
print('\n=== UPDATE 2: Flag Negative Delivery Times ===')
update_result2 = db.delivery_operations.update_many(
    {'data_quality_flags.negative_delivery_time': True},
    {
        '$set': {
            'data_quality_flags.requires_review': True,
            'data_quality_flags.review_reason': 'Completion timestamp precedes dispatch - likely system error'
        }
    }
)
print(f'Matched: {update_result2.matched_count}, Modified: {update_result2.modified_count}')

# Verify update
flagged = db.delivery_operations.find_one({'data_quality_flags.requires_review': True})
print('\nVerification - flagged document data_quality_flags:')
pprint(flagged['data_quality_flags'])

In [ ]:
# --- DELETE Operations ---

# D1: Remove test/orphan events (events with no linked customer in customers table)
print('=== DELETE: Count orphan events before removal ===')
valid_customers = set(customers['customer_id'].tolist())
orphan_count = db.app_event_stream.count_documents(
    {'customer_id': {'$nin': list(valid_customers)}}
)
print(f'Orphan events found: {orphan_count}')

if orphan_count > 0:
    delete_result = db.app_event_stream.delete_many(
        {'customer_id': {'$nin': list(valid_customers)}}
    )
    print(f'Deleted: {delete_result.deleted_count}')
else:
    print('No orphan events to delete - all events link to valid customers.')

In [ ]:
# ============================================================
# AGGREGATION PIPELINE 1: Hub Failure Analysis
# Purpose: Replicate the complex SQL hub dashboard query using
# MongoDB aggregation, demonstrating pipeline stage composition.
# ============================================================

print('=== Aggregation Pipeline 1: Hub Performance Dashboard ===')

pipeline_hub = [
    # Stage 1: Group by hub
    {'$group': {
        '_id': '$hub.hub_name',
        'hub_type': {'$first': '$hub.hub_type'},
        'zone': {'$first': '$hub.zone'},
        'capacity_score': {'$first': '$hub.capacity_score'},
        'total_deliveries': {'$sum': 1},
        'failed': {'$sum': {'$cond': [{'$eq': ['$delivery_status', 'Failed']}, 1, 0]}},
        'delayed': {'$sum': {'$cond': [{'$eq': ['$delivery_status', 'Delayed']}, 1, 0]}},
        'total_cost': {'$sum': '$route.fuel_or_charge_cost'},
        'avg_distance': {'$avg': '$route.distance_km'},
        'avg_overrides': {'$avg': '$route.manual_override_count'},
        'total_incidents': {'$sum': {'$size': '$incidents'}}
    }},
    # Stage 2: Add computed fields
    {'$addFields': {
        'failure_rate_pct': {'$round': [{'$multiply': [{'$divide': ['$failed', '$total_deliveries']}, 100]}, 1]},
        'delay_rate_pct': {'$round': [{'$multiply': [{'$divide': ['$delayed', '$total_deliveries']}, 100]}, 1]},
        'avg_cost_per_delivery': {'$round': [{'$divide': ['$total_cost', '$total_deliveries']}, 2]}
    }},
    # Stage 3: Sort by failure rate
    {'$sort': {'failure_rate_pct': -1}}
]

results = list(db.delivery_operations.aggregate(pipeline_hub))
for r in results:
    print(f"{r['_id']:20s} | Zone: {r['zone']:10s} | Type: {r['hub_type']:10s} | "
          f"Fail: {r['failure_rate_pct']}% | Delay: {r['delay_rate_pct']}% | "
          f"Incidents: {r['total_incidents']} | Overrides: {r['avg_overrides']:.2f}")

print('\nThis pipeline replaces a 5-table SQL JOIN with a single collection scan.')

In [ ]:
# ============================================================
# AGGREGATION PIPELINE 2: Customer Risk Segmentation
# Purpose: Identify at-risk customer segments by combining
# complaint frequency, order value, and loyalty score.
# ============================================================

print('=== Aggregation Pipeline 2: Customer Risk Segmentation ===')

pipeline_risk = [
    # Stage 1: Add risk tier classification
    {'$addFields': {
        'risk_tier': {
            '$switch': {
                'branches': [
                    {'case': {'$and': [
                        {'$gte': ['$metrics.total_complaints', 3]},
                        {'$lt': ['$engagement.loyalty_score', 40]}
                    ]}, 'then': 'Critical'},
                    {'case': {'$and': [
                        {'$gte': ['$metrics.total_complaints', 2]},
                        {'$lt': ['$engagement.loyalty_score', 50]}
                    ]}, 'then': 'High'},
                    {'case': {'$gte': ['$metrics.total_complaints', 1]}, 'then': 'Medium'},
                ],
                'default': 'Low'
            }
        }
    }},
    # Stage 2: Group by risk tier and zone
    {'$group': {
        '_id': {'risk_tier': '$risk_tier', 'zone': '$profile.home_zone'},
        'customer_count': {'$sum': 1},
        'avg_loyalty': {'$avg': '$engagement.loyalty_score'},
        'avg_order_value': {'$avg': '$metrics.total_order_value'},
        'avg_complaints': {'$avg': '$metrics.total_complaints'},
        'total_compensation': {'$sum': '$metrics.total_compensation'}
    }},
    # Stage 3: Sort
    {'$sort': {'_id.risk_tier': 1, 'customer_count': -1}}
]

results = list(db.customer_journeys.aggregate(pipeline_risk))
for r in results:
    if r['_id']['risk_tier'] in ['Critical', 'High']:
        print(f"Risk: {r['_id']['risk_tier']:8s} | Zone: {r['_id']['zone']:10s} | "
              f"Customers: {r['customer_count']:3d} | Avg Loyalty: {r['avg_loyalty']:.1f} | "
              f"Compensation: £{r['total_compensation']:.2f}")

In [ ]:
# ============================================================
# AGGREGATION PIPELINE 3: App Event Performance Analysis
# Purpose: Analyse app platform health by event type, device,
# and zone to identify technical hotspots.
# ============================================================

print('=== Aggregation Pipeline 3: App Platform Health ===')

pipeline_app = [
    # Stage 1: Group by event type and device
    {'$group': {
        '_id': {'event_type': '$event_type', 'device': '$device_type'},
        'total_events': {'$sum': 1},
        'failures': {'$sum': {'$cond': [{'$eq': ['$success', False]}, 1, 0]}},
        'avg_latency': {'$avg': '$api_latency_ms'},
        'p95_latency': {'$percentile': {'input': '$api_latency_ms', 'p': [0.95], 'method': 'approximate'}},
        'high_latency_count': {'$sum': {'$cond': ['$is_high_latency', 1, 0]}}
    }},
    # Stage 2: Compute failure rate
    {'$addFields': {
        'failure_rate': {'$round': [{'$multiply': [{'$divide': ['$failures', '$total_events']}, 100]}, 1]},
        'high_latency_pct': {'$round': [{'$multiply': [{'$divide': ['$high_latency_count', '$total_events']}, 100]}, 1]}
    }},
    # Stage 3: Sort by failure rate
    {'$sort': {'failure_rate': -1, 'avg_latency': -1}}
]

results = list(db.app_event_stream.aggregate(pipeline_app))
for r in results:
    print(f"{r['_id']['event_type']:30s} | Device: {r['_id']['device']:8s} | "
          f"Events: {r['total_events']:4d} | Fail: {r['failure_rate']}% | "
          f"Avg Latency: {r['avg_latency']:.0f}ms | High Latency: {r['high_latency_pct']}%")

print('\nchat_escalated and payment_retry are the only event types with failures.')
print('This aggregation pipeline replaces what would require complex SQL GROUP BY with CASE statements.')

In [ ]:
# ============================================================
# AGGREGATION PIPELINE 4: Cross-System Mismatch Detection
# Purpose: Use MongoDB to identify deliveries marked OnTime
# that have incidents - confirming the data mismatch issue.
# ============================================================

print('=== Aggregation Pipeline 4: Data Mismatch Detection ===')

pipeline_mismatch = [
    # Only OnTime deliveries
    {'$match': {'delivery_status': 'OnTime', 'has_incidents': True}},
    # Unwind incidents for analysis
    {'$unwind': '$incidents'},
    # Group by incident type
    {'$group': {
        '_id': '$incidents.incident_type',
        'count': {'$sum': 1},
        'affected_deliveries': {'$addToSet': '$delivery_id'},
        'avg_severity_critical_high': {
            '$sum': {'$cond': [{'$in': ['$incidents.severity', ['Critical', 'High']]}, 1, 0]}
        }
    }},
    # Add affected count
    {'$addFields': {
        'affected_delivery_count': {'$size': '$affected_deliveries'}
    }},
    {'$project': {'affected_deliveries': 0}},
    {'$sort': {'count': -1}}
]

results = list(db.delivery_operations.aggregate(pipeline_mismatch))
total_mismatched = 0
for r in results:
    print(f"{r['_id']:20s} | Incidents: {r['count']:3d} | "
          f"Unique Deliveries: {r['affected_delivery_count']:3d} | "
          f"Critical/High: {r['avg_severity_critical_high']}")
    total_mismatched += r['affected_delivery_count']

print(f'\nTotal OnTime deliveries with at least one incident: confirms cross-system mismatch.')
print('This is the key evidence that NorthStar needs unified data architecture.')

In [ ]:
# ============================================================
# COLLECTION STATISTICS
# ============================================================

print('=== Collection Statistics ===')
for coll_name in ['customer_journeys', 'delivery_operations', 'app_event_stream']:
    coll = db[coll_name]
    stats = db.command('collStats', coll_name)
    print(f'\n{coll_name}:')
    print(f'  Documents: {stats["count"]}')
    print(f'  Avg document size: {stats.get("avgObjSize", 0)} bytes')
    print(f'  Indexes: {stats["nindexes"]}')
    print(f'  Index names: {list(coll.index_information().keys())}')

# Clean up
# client.close()
print('\nMongoDB operations complete.')

## MongoDB Development: Key Findings

### Document Model Advantages for NorthStar

1. **Customer Journey documents** embed orders, complaints, and app activity in a single document. What required a 4-table SQL JOIN now takes one `find()` call, improving both query speed and code simplicity.

2. **Delivery Operation documents** combine delivery context (driver, vehicle, hub, incidents) into a single document. The operations team can see the full picture of any delivery without cross-referencing multiple tables.

3. **App Event Stream** is naturally suited to MongoDB's schema-flexible document model, since event types have varying attributes. The denormalised zone and customer data enable efficient aggregation without lookups.

### Aggregation Pipeline Results
- **Hub performance** pipeline confirms H08 Midtown Relay and H05 Central Core as worst performers
- **Customer risk segmentation** identifies Critical and High-risk customers by zone
- **App platform health** reveals chat_escalated (50% failure) and payment_retry (27.5% failure) as problem areas
- **Cross-system mismatch** pipeline confirms 191 incidents on OnTime deliveries, validating the case study's concern about fragmented systems

### Indexing Strategy
- Single-field indexes on frequently queried fields (customer_id, delivery_status, event_type)
- Compound index on `(delivery_status, hub.hub_id)` for the hub performance pipeline
- Compound index on `(event_type, success)` for the app health pipeline